# PyPSA Network Viewer - Example Notebook

**Version 0.2.0** - Complete interactive network visualization

This notebook demonstrates all features:
1. ✅ Basic HTML viewer
2. ✅ Network Summary, Global Constraints, and Custom Plots
3. ✅ Complete viewer with all features
4. ✅ Excel template generator

## Setup and Import

**⚠️ IMPORTANT:** This cell forces a reload of the module to ensure you're using the latest version with all bug fixes.

In [ ]:
import pypsa
import plotly.graph_objects as go
from pypsa_network_viewer import html_network, generate_template

print("✓ Imports ready")

## Load and Optimize Network

In [2]:
# Load example network
network = pypsa.examples.ac_dc_meshed()
print(f"Loaded: {network.name}")
print(f"Buses: {len(network.buses)}, Generators: {len(network.generators)}")

INFO:pypsa.network.io:Retrieving network data from https://github.com/PyPSA/PyPSA/raw/v1.0.2/examples/networks/ac-dc-meshed/ac-dc-meshed.nc.
INFO:pypsa.network.io:New version 1.0.3 available! (Current: 1.0.2)
INFO:pypsa.network.io:Imported network 'AC-DC-Meshed' has buses, carriers, generators, global_constraints, lines, links, loads


Loaded: AC-DC-Meshed
Buses: 9, Generators: 6


In [3]:
# Optimize
status = network.optimize()
print(f"Status: {status[0]}, Objective: {network.objective:.2f}")

Index(['2', '3', '4'], dtype='object', name='name')
Index(['0', '1', '5', '6'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.19s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 188 primals, 468 duals
Objective: -3.47e+06
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, Line-ext-s-lower, Line-ext-s-upper, Link-ext-p-lower, Link-ext-p-upper, Kirchhoff-Voltage-Law were not assigned to the network.


Status: ok, Objective: -3474256.04


## Example 1: Basic Viewer

In [ ]:
html_network(network, file_name='basic.html')

Network analyzer generated: basic.html
Components found: ['generators', 'links', 'loads', 'lines', 'buses']


'basic.html'

## Example 2: Create Custom Plots

In [5]:
# Generator output
fig1 = go.Figure()
for col in network.generators_t.p.columns:
    fig1.add_trace(go.Scatter(x=network.generators_t.p.index, y=network.generators_t.p[col], mode='lines', name=col))
fig1.update_layout(title='Generator Power Output', xaxis_title='Time', yaxis_title='MW')

# Link flow
fig2 = go.Figure()
for col in network.links_t.p0.columns:
    fig2.add_trace(go.Scatter(x=network.links_t.p0.index, y=network.links_t.p0[col], mode='lines', name=col))
fig2.update_layout(title='Link Power Flow', xaxis_title='Time', yaxis_title='MW')

# Bus prices
fig3 = go.Figure()
for col in network.buses_t.marginal_price.columns:
    fig3.add_trace(go.Scatter(x=network.buses_t.marginal_price.index, y=network.buses_t.marginal_price[col], mode='lines', name=col))
fig3.update_layout(title='Bus Marginal Prices', xaxis_title='Time', yaxis_title='$/MWh')

# Total generation
total = network.generators_t.p.sum()
fig4 = go.Figure([go.Bar(x=total.index, y=total.values, marker_color='lightblue')])
fig4.update_layout(title='Total Energy by Generator', xaxis_title='Generator', yaxis_title='MWh')

custom_figs = [fig1, fig2, fig3, fig4]
print(f"Created {len(custom_figs)} custom plots")

Created 4 custom plots


## Example 3: Complete Viewer with All Features

In [6]:
output = html_network(
    network,
    file_path='output',
    file_name='complete_viewer.html',
    title='Complete Network Analysis',
    currency='$',
    custom_plots=custom_figs
)

print("\n" + "="*60)
print(f"Complete viewer: {output}")
print("="*60)
print("\nIncludes:")
print("  ✓ Network Summary")
print("  ✓ Global Constraints")
print(f"  ✓ {len(custom_figs)} Custom Plots")
print("  ✓ All component data")
print("\nOpen in browser to explore!")

Network analyzer generated: output\complete_viewer.html
Components found: ['generators', 'links', 'loads', 'lines', 'buses']

Complete viewer: output\complete_viewer.html

Includes:
  ✓ Network Summary
  ✓ Global Constraints
  ✓ 4 Custom Plots
  ✓ All component data

Open in browser to explore!


## Example 4: Excel Template Generator

Generate a pre-configured Excel workbook that users can fill in to define a PyPSA network.
The template includes all component sheets, dropdown validations, and a snapshots sheet.

Key parameters:
- `start_year` / `start_month` — when the simulation starts
- `years_duration` / `months_duration` / `days_duration` — exactly one must be set
- `resolution_str` — timestep size (`'H'`, `'2H'`, `'4H'`, `'6H'`, `'8H'`, `'0.5H'`, `'15m'`, `'30m'`)
- `link_outputs` — number of link output buses (bus0 … bus_N)
- `process_outputs` — number of process output buses (bus0 … bus_N)

In [ ]:
# --- 1 year, hourly resolution -------------------------------------------
generate_template(
    output_name='network_template_1y.xlsx',
    start_year=2025,
    start_month=1,
    years_duration=1,
    resolution_str='H',
    drop_leap_day=True,
    link_outputs=1,
    process_outputs=2,
)

In [ ]:
# --- 3 months, 30-minute resolution --------------------------------------
generate_template(
    output_name='network_template_3m_30min.xlsx',
    start_year=2025,
    start_month=6,
    months_duration=3,
    resolution_str='30m',
    drop_leap_day=False,
    link_outputs=2,   # bus0, bus1, bus2
    process_outputs=2,
)

## Usage Tips

### Viewing HTML Files:
1. Open HTML in any browser
2. Select from dropdowns
3. Click "Load Data"
4. Interact with plots (zoom, pan, hover)

### Available Views:
- **Network Summary** - Properties and optimization results
- **Global Constraints** - Constraints with dual values
- **Custom Plots** - Your Plotly visualizations
- **Components** - All static and timeseries data

### Components:
- Generators, Links, Loads
- Stores, Storage Units
- Lines, Transformers, Buses

For more info: [README.md](README.md) | [EXAMPLE_USAGE.md](EXAMPLE_USAGE.md)